# Tabular Neural Network (TabNet) for tabular or structured dataset

In [ ]:
# ----------------- Basic Libraries -----------------
import os
import random
import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd

# ----------------- Geospatial Libraries -----------------
from pyproj import Proj
from shapely.geometry import Point
from geopandas import GeoDataFrame

# ----------------- Sklearn Libraries -----------------
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, QuantileTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score, make_scorer

# ----------------- PyTorch Libraries -----------------
import torch
from torch import nn
from torch.optim import Adam

# ----------------- TabNet Library -----------------
from pytorch_tabnet.tab_model import TabNetRegressor

# ----------------- Statistical Libraries -----------------
from scipy.stats import shapiro
from scipy.stats import skew  # Better than using pandas just for skew

myProj = Proj("+proj=utm +zone=43 +north +datum=WGS84 +units=m +no_defs")



In [ ]:
df=pd.read_csv('Train_Test_REE_final.csv')
df.dropna(inplace=True)
df

In [ ]:
# Inverse projection to get Long, Lat from UTM coordinates. Then we define WGS 1984(also called EPSG:4326) coordinate reference system(crs)
long,lat=myProj(df.X.values,df.Y.values, inverse=True)
geometry = [Point(xy) for xy in zip(long, lat)]                  # All locations converted to point geometry
df = GeoDataFrame(df, crs="EPSG:4326", geometry=geometry)        # Dataframe converted to Geospatial compatible dataframe(Geodataframe)
df

# Splitting dataframe for visualization

In [ ]:
# Split the dataframe into training (80%) and testing (20%) sets randomly
df_tr, df_ts = train_test_split(df, test_size=0.2, random_state=42)

## Preprocessing and again spilliting of scaled datasets

In [ ]:
# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
X = df.iloc[:,3:-1].values
np.shape(X)
y = df['Prob'].values

In [ ]:
# Enhanced Data Preprocessing
def preprocess_data(X, y):
    """Enhanced data preprocessing pipeline"""
    # Feature scaling
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Ensure y is 2D array (required by TabNet)
    y_reshaped = y.reshape(-1, 1)  # <-- just reshape, no `.values`
    
    # Target transformation if needed
    if (y < 0).any() or (abs(skew(y.flatten())) > 1.0):  # <-- use scipy.stats.skew
        target_scaler = QuantileTransformer(output_distribution='normal', random_state=SEED)
        y_transformed = target_scaler.fit_transform(y_reshaped)
    else:
        target_scaler = None
        y_transformed = y_reshaped

    return X_scaled, y_transformed, scaler, target_scaler

# Apply preprocessing
X_scaled, y_transformed, scaler, target_scaler = preprocess_data(X, y)

In [ ]:
# Split data (stratify if classification, shuffle=False for time series)
X_tr, X_ts, y_tr, y_ts = train_test_split(
    X_scaled, y_transformed, 
    test_size=0.2, 
    random_state=SEED,
    #shuffle=False  # Important for time series
)

## Define the TabNet model and choosing best model parameters

In [ ]:
# Debugged TabNet Model Wrapper
class DebuggedTabNetWrapper:
    def __init__(self, n_d=64, n_a=64, n_steps=5, gamma=1.5, 
                 lambda_sparse=1e-4, momentum=0.02, batch_size=256,
                 max_epochs=200, lr=0.02):
        self.n_d = n_d
        self.n_a = n_a
        self.n_steps = n_steps
        self.gamma = gamma
        self.lambda_sparse = lambda_sparse
        self.momentum = momentum
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.lr = lr
        self.model = None
        self._initialize_model()

    def _initialize_model(self):
        self.model = TabNetRegressor(
            n_d=self.n_d,
            n_a=self.n_a,
            n_steps=self.n_steps,
            gamma=self.gamma,
            lambda_sparse=self.lambda_sparse,
            momentum=self.momentum,
            seed=SEED,
            optimizer_params=dict(lr=self.lr),
            verbose=0,
            scheduler_params={"step_size": 10, "gamma": 0.9},
            scheduler_fn=torch.optim.lr_scheduler.StepLR
        )

    def fit(self, X, y, eval_set=None, **kwargs):
        # Ensure y is 2D
        if y.ndim == 1:
            y = y.reshape(-1, 1)
        
        # Setup eval_set if provided
        if eval_set:
            eval_X, eval_y = eval_set
            if eval_y.ndim == 1:
                eval_y = eval_y.reshape(-1, 1)
            eval_set = [(eval_X, eval_y)]
        else:
            eval_set = None

        self.model.fit(
            X_train=X,
            y_train=y,
            eval_set=eval_set,
            eval_metric=['rmse', 'mae'],
            max_epochs=self.max_epochs,
            patience=20 if eval_set else 0,  # Early stopping if eval_set
            batch_size=self.batch_size,
            virtual_batch_size=128,
            **kwargs
        )
        return self

    def predict(self, X):
        return self.model.predict(X)

    def get_params(self, deep=True):
        return {
            'n_d': self.n_d,
            'n_a': self.n_a,
            'n_steps': self.n_steps,
            'gamma': self.gamma,
            'lambda_sparse': self.lambda_sparse,
            'momentum': self.momentum,
            'batch_size': self.batch_size,
            'max_epochs': self.max_epochs,
            'lr': self.lr
        }

    def set_params(self, **params):
        for param, value in params.items():
            setattr(self, param, value)
        self._initialize_model()
        return self

# Adjusted Hyperparameter Search Space
param_grid = {
    'n_d': [32, 64],
    'n_a': [32, 64],
    'n_steps': [3, 5],
    'gamma': [1.2, 1.5],
    'lambda_sparse': [1e-5, 1e-4],
    'momentum': [0.01, 0.02],
    'batch_size': [128, 256],
    'max_epochs': [100, 150],
    'lr': [0.01, 0.02]
}

# Custom Scoring with Safe Error Handling
def safe_r2_scorer(estimator, X, y):
    try:
        if y.ndim == 1:
            y = y.reshape(-1, 1)
        estimator.fit(X, y)
        y_pred = estimator.predict(X)
        return r2_score(y, y_pred)
    except Exception as e:
        print(f"Error during scoring: {str(e)}")
        return -np.inf

scorers = {
    'R2': make_scorer(r2_score),
    'MSE': make_scorer(mean_squared_error, greater_is_better=False),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'Explained_Variance': make_scorer(explained_variance_score)
}

# Prepare Your Data
# You must define X_tr, X_ts, y_tr, y_ts yourself here.
# Example:
# X_tr, X_ts, y_tr, y_ts = train_test_split(X, y, test_size=0.2, random_state=SEED)

# If you used target scaling
target_scaler = None  # or your actual scaler

#  Run Debugged Randomized Search
model = DebuggedTabNetWrapper()

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=20,
    cv=3,
    scoring=scorers,
    refit='R2',
    verbose=2,
    random_state=SEED,
    n_jobs=1,
    error_score='raise'
)

print("Starting debugged hyperparameter search...")
start_time = time.time()
try:
    random_search.fit(X_tr, y_tr)
except Exception as e:
    print(f"Error during search: {str(e)}")
    raise
end_time = time.time()
print(f"Search completed in {(end_time - start_time)/60:.2f} minutes")

# Evaluation with Proper Target Reshaping
if hasattr(random_search, 'best_estimator_'):
    best_model = random_search.best_estimator_.model
    
    y_pred_train = best_model.predict(X_tr)
    y_pred_test = best_model.predict(X_ts)
    
    # Inverse transform if target was scaled
    if target_scaler:
        y_tr = target_scaler.inverse_transform(y_tr)
        y_ts = target_scaler.inverse_transform(y_ts)
        y_pred_train = target_scaler.inverse_transform(y_pred_train)
        y_pred_test = target_scaler.inverse_transform(y_pred_test)
    
    y_tr = y_tr.flatten()
    y_ts = y_ts.flatten()
    y_pred_train = y_pred_train.flatten()
    y_pred_test = y_pred_test.flatten()
    
    metrics = {
        'Train MSE': mean_squared_error(y_tr, y_pred_train),
        'Test MSE': mean_squared_error(y_ts, y_pred_test),
        'Train MAE': mean_absolute_error(y_tr, y_pred_train),
        'Test MAE': mean_absolute_error(y_ts, y_pred_test),
        'Train R2': r2_score(y_tr, y_pred_train),
        'Test R2': r2_score(y_ts, y_pred_test),
        'Train Explained Variance': explained_variance_score(y_tr, y_pred_train),
        'Test Explained Variance': explained_variance_score(y_ts, y_pred_test)
    }
    
    print("\nBest Parameters:", random_search.best_params_)
    print("Best R2 Score:", random_search.best_score_)
    print("\nFinal Performance Metrics:")
    for name, value in metrics.items():
        print(f"{name}: {value:.4f}")
    
    # 8. Diagnostic Plots
    def plot_diagnostics(y_true, y_pred, title):
        residuals = y_true - y_pred
        
        plt.figure(figsize=(15, 5))
        
        # Actual vs Predicted
        plt.subplot(1, 3, 1)
        plt.scatter(y_true, y_pred, alpha=0.5)
        plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], 'r--')
        plt.xlabel('Actual')
        plt.ylabel('Predicted')
        plt.title(f'Actual vs Predicted: {title}')
        
        # Residuals
        plt.subplot(1, 3, 2)
        plt.scatter(y_pred, residuals, alpha=0.5)
        plt.axhline(y=0, color='r', linestyle='--')
        plt.xlabel('Predicted')
        plt.ylabel('Residuals')
        plt.title(f'Residuals: {title}')
        
        # Distribution
        plt.subplot(1, 3, 3)
        plt.hist(residuals, bins=30, alpha=0.7)
        plt.xlabel('Residuals')
        plt.ylabel('Frequency')
        plt.title(f'Residual Distribution: {title}')
        
        plt.tight_layout()
        plt.show()
    
    plot_diagnostics(y_tr, y_pred_train, "Training Set")
    plot_diagnostics(y_ts, y_pred_test, "Test Set")
    
    # Feature importance
    if hasattr(best_model, 'feature_importances_'):
        feature_importances = best_model.feature_importances_
        sorted_idx = np.argsort(feature_importances)[::-1]
        
        plt.figure(figsize=(10, 6))
        plt.bar(range(len(feature_importances)), feature_importances[sorted_idx])
        plt.title("Feature Importances")
        plt.tight_layout()
        plt.show()
else:
    print("Hyperparameter search failed. Please check error messages above.")


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score

# Extract the best parameters from RandomizedSearchCV
best_params = random_search.best_params_

# Final model (already the best from RandomizedSearchCV)
final_wrapper = random_search.best_estimator_

# Extract the underlying TabNet model
final_model = final_wrapper.model

# Refit the final model on full training data
# (In case you want full training without CV splits)

# Make sure y_tr is 2D
if y_tr.ndim == 1:
    y_tr = y_tr.reshape(-1, 1)

# Fit the model
final_model.fit(
    X_train=X_tr,
    y_train=y_tr,
    eval_set=[(X_ts, y_ts.reshape(-1, 1))],  # Validation set
    eval_metric=['rmse', 'mae'],
    max_epochs=best_params['max_epochs'],  # Use best max_epochs
    patience=20,  # Early stopping patience
    batch_size=best_params['batch_size'],
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False
)

# Predict on training data
y_train_pred = final_model.predict(X_tr)
train_mse = mean_squared_error(y_tr.flatten(), y_train_pred.flatten())
print(f'Training Data (Known to Machine & Human) - Mean Squared Error: {train_mse:.4f}')

# Predict on test data
y_test_pred = final_model.predict(X_ts)
test_mse = mean_squared_error(y_ts.flatten(), y_test_pred.flatten())
print(f'Test Data (Unknown to Machine) - Mean Squared Error: {test_mse:.4f}')

# Additional Metrics (optional)
train_r2 = r2_score(y_tr.flatten(), y_train_pred.flatten())
test_r2 = r2_score(y_ts.flatten(), y_test_pred.flatten())

print(f'Training R2 Score: {train_r2:.4f}')
print(f'Test R2 Score: {test_r2:.4f}')


In [ ]:
# Model performance
print('Training data(Known to machine, human)')
mse = mean_squared_error(y_tr, y_train_pred)
rmse = mse**0.5
print(f"Training MSE: {mean_squared_error(y_tr, y_train_pred)}")
print(f"Training RMSE: {rmse}")
print(f"Training MAE: {mean_absolute_error(y_tr, y_train_pred)}")
print(f"Training R2 Score: {r2_score(y_tr, y_train_pred)}")
print(f"Training Explained Variance: {explained_variance_score(y_tr, y_train_pred)}")


print('Test data(Unknown to machine, known to human)')
mse = mean_squared_error(y_ts, y_test_pred)
rmse = mse**0.5
print(f"Test MSE: {mean_squared_error(y_ts, y_test_pred)}")
print(f"Training RMSE: {rmse}")
print(f"Test MAE: {mean_absolute_error(y_ts, y_test_pred)}")
print(f"Test R2 Score: {r2_score(y_ts, y_test_pred)}")
print(f"Test Explained Variance: {explained_variance_score(y_ts, y_test_pred)}")

In [ ]:
df

In [ ]:
X = df.iloc[:,3:-1].values
np.shape(X)
X = MinMaxScaler().fit_transform(X)
X

In [ ]:
#X=df.iloc[:,2:-2].values
all_pred = final_model.predict(X) 


In [ ]:
y=df['Prob'].values
print('Train+Test data Mean squared error=', mean_squared_error(y, all_pred))
df['Predicted_Probability']=all_pred

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure model is on CPU
device = torch.device('cpu')
final_model.network.to(device)
final_model.network.eval()

# Convert X_ts to torch tensor
X_tensor = torch.tensor(X_ts, dtype=torch.float32, device=device, requires_grad=True)

# Forward pass
y_pred, _ = final_model.network(X_tensor)  # <-- FIXED: unpack the tuple

# Compute gradients
y_pred.sum().backward()

# Get gradients
gradients = X_tensor.grad.detach().numpy()

# Compute feature importance
feat_importance = np.abs(gradients).mean(axis=0)

# Ensure column names are correct
X_ts_df = pd.DataFrame(X_ts, columns=df.iloc[:, 3:-2].columns)

# Pandas Series
feat_importances = pd.Series(feat_importance, index=X_ts_df.columns)

# Plot
plt.figure(figsize=(10, 6))
feat_importances.nlargest(10).plot(kind='barh', color='skyblue')
plt.xlabel("Feature Importance Score")
plt.ylabel("Features")
plt.title("Gradient-based Feature Importance in TabNet")
plt.gca().invert_yaxis()
plt.show()

# Print
print("Top 10 Important Features:\n", feat_importances.nlargest(10))


In [ ]:
mdf=pd.read_csv('Master_REE.csv')
mdf.dropna(inplace=True)
mdf

In [ ]:
MX=mdf.iloc[:,2:].values
MX

In [ ]:
MX=mdf.iloc[:,2:].values
MX_scale = MinMaxScaler().fit_transform(MX)
all_pred = final_model.predict(MX_scale) 
mdf['Predicted_Probability']=all_pred

In [ ]:
mdf.to_csv('REE_TabNet.csv', index=False)